Point 3: NER & Keyphrase Extraction

Read the text from the PDF and automatically find:
- Important named things (like Vitamin C, Fish, Red meat), done with NER (Named Entity Recognition);
- Important phrases or concepts that might not match exactly the KB (like iodised salt, wholegrain rye bread), done with keyphrase extraction.

NER: using spaCy library (can mark words or phrases as certain types). Using an EntityRuler that contains patterns like "Vitamin C", "red meat", then running the model over the text chunks extracted from the PDF and contained in the raw_text.json. spaCy will label words it recognizes with custom categories.

Keyphrase extraction: use libraries like YAKE (finds keywords just by analyzing word frequency and patterns) and KeyBERT (finds key phrases that are semantically important using a transformer), feed each text chunk to YAKE or KeyBERT and it will give you the most important phrases. Then compare these phrases with your KB aliases, if a phrase isn’t already in your KB, note it down — it could become a new alias or even a new entity later.

Cell 1 — Imports, configuration, and helper functions
This cell loads your Python libraries, defines file paths, and adds small helper functions for normalization and validation.
You only need to run this once at the top of your notebook.

In [ ]:
# --- Point 3 · Step 1: setup environment & helpers -------------------------

from __future__ import annotations
import json, re
from pathlib import Path
from typing import Dict, List, Iterable, Any
from collections import defaultdict

# --- Configuration: adjust if your files are in another folder -------------
KB_PATH = Path("kb-2.json")            # knowledge base (canonical entities + aliases)
RAW_TEXT_PATH = Path("raw_text.jsonl") # text chunks extracted from PDF
ONTOLOGY_PATH = Path("ontologyv2.yaml")# ontology schema
DEFAULT_DOC_ID = "SUSTAINABLE_HEALTH_FROM_FOOD"

# --- Helper functions ------------------------------------------------------

def norm(text: str) -> str:
    """Normalize surface forms for alias lookup (lowercase, collapse whitespace & punctuation)."""
    t = text.lower().strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[‐–—−]", "-", t)
    t = re.sub(r"[“”]", '"', t).replace("’","'")
    return t.strip()

def validate_kb_entry(e: Dict[str, Any]) -> List[str]:
    """Light validation for one KB entry."""
    errs = []
    if not {"id","label","class"} <= e.keys():
        errs.append("Missing required keys (id/label/class).")
    else:
        if not re.match(r"^ex:[a-zA-Z][a-zA-Z0-9_]*\.[a-z0-9_]+$", e["id"]):
            errs.append(f"Bad id format: {e['id']}")
    if "aliases" in e and not isinstance(e["aliases"], list):
        errs.append("aliases must be a list.")
    return errs


Cell 2 — Load and validate your KB
This cell reads kb-2.json, removes duplicate aliases, validates IDs, and reports how many entities you have.
It also builds two key structures you’ll use later:
entityruler_patterns for spaCy’s EntityRuler
alias_index for linking spans to canonical IDs

In [ ]:
# --- Load KB & basic validation --------------------------------------------

def load_kb(kb_path: Path) -> List[Dict[str, Any]]:
    data = json.loads(kb_path.read_text(encoding="utf-8"))
    for e in data:
        seen, dedup = set(), []
        for a in e.get("aliases", []):
            if a and a not in seen:
                seen.add(a); dedup.append(a)
        e["aliases"] = dedup
    return data

def build_entityruler_patterns(kb: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """Return patterns ready to add to spaCy EntityRuler."""
    pats = []
    for ent in kb:
        label = ent["class"].upper()
        for s in [ent["label"], *ent["aliases"]]:
            if s and s.strip():
                pats.append({"label": label, "pattern": s})
    return pats

def build_alias_index(kb: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, str]]]:
    """normalized surface -> list of {id,class,label} dicts."""
    idx = defaultdict(list)
    for ent in kb:
        for s in [ent["label"], *ent["aliases"]]:
            key = norm(s)
            if key:
                idx[key].append({"id": ent["id"], "class": ent["class"], "label": ent["label"]})
    return idx

# --- Execute load ----------------------------------------------------------
kb = load_kb(KB_PATH)
entityruler_patterns = build_entityruler_patterns(kb)
alias_index = build_alias_index(kb)

print(f"KB entries loaded: {len(kb)}")
print(f"EntityRuler patterns: {len(entityruler_patterns)}")
print(f"Alias index entries: {len(alias_index)}")

# Quick check for malformed entries
errors = []
for e in kb:
    errs = validate_kb_entry(e)
    if errs:
        errors.append((e["id"], errs))
if errors:
    print(f"[WARN] {len(errors)} KB entries with issues (showing 3):", errors[:3])
else:
    print("KB validation: OK")


Cell 3 — Define label↔class map, over-broad terms, and context cues
This small config keeps your ontology classes aligned with NER labels,
and prepares guardrails for generic terms like “sugar” or “fish”.

In [ ]:
# --- Mappings and guardrails ----------------------------------------------

LABEL_TO_CLASS = {
    "NUTRIENT": "nutrient",
    "FOOD_GROUP": "foodGroup",
    "FOOD_ITEM": "foodItem",
    "TECHNIQUE": "technique",
}
CLASS_TO_LABEL = {v:k for k,v in LABEL_TO_CLASS.items()}

# Over-broad terms that might match too often (you’ll filter later)
OVERBROAD = {"sugar","protein","fish","meat","fat","fats","oil","oils",
             "juice","juices","drink","drinks","salt"}

# Context cues for validating those generic matches in Step 4
CONTEXT_CUES = {
    "nutrient": {"intake","rda","ai","ul","e%","deficiency","vitamin","mineral"},
    "foodGroup": {"portion","g/day","g/week","eat","limit","increase","reduce","serving"},
    "foodItem": {"portion","eat","limit","increase","reduce","serving"},
    "technique": {"fried","boiled","baked","cooked","prepared","processing"},
}

print("Mappings and guardrails ready.")


Cell 4 — Prepare iterator over raw_text.jsonl
This cell streams your extracted text chunks one by one so you don’t load the whole file into memory.
It also fills missing docId fields with a default value.

In [ ]:
# --- Iterator for raw_text.jsonl ------------------------------------------

def iter_raw_text(path: Path, default_doc_id: str):
    """Yield dicts with docId, page, section, text (skip invalid lines)."""
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = obj.get("text")
            page = obj.get("page")
            section = obj.get("section", "")
            if text is None or page is None:
                continue
            doc_id = obj.get("docId", default_doc_id)
            yield {
                "docId": doc_id,
                "page": page,
                "section": section,
                "text": text,
            }

# Build the iterator and preview a couple of chunks
raw_stream = iter_raw_text(RAW_TEXT_PATH, DEFAULT_DOC_ID)

first_two = []
for _ in range(2):
    try:
        first_two.append(next(raw_stream))
    except StopIteration:
        break

print("Sample text chunks:")
for i, rec in enumerate(first_two, 1):
    print(f"[{i}] page={rec['page']} section='{rec['section']}'")
    print("   text:", rec['text'][:120].replace("\n"," "), "...\n")

# Re-chain the first two back into the iterator for later steps
def chain_iters(first, rest_iter):
    for x in first:
        yield x
    for x in rest_iter:
        yield x

raw_stream = chain_iters(first_two, iter_raw_text(RAW_TEXT_PATH, DEFAULT_DOC_ID))


Cell 5 — Initialize output buffers and summary
This final setup cell creates empty lists for linked/unlinked mentions and prints a quick summary.
These structures will be filled in Steps 3–6 of your pipeline.

In [ ]:
# --- Output scaffolding and summary ---------------------------------------

mentions_buffer: List[Dict[str, Any]] = []   # linked mentions (filled later)
debug_unlinked: List[Dict[str, Any]] = []    # unlinked mentions for KB growth

kb_classes = sorted({e["class"] for e in kb})

print("=== Step 1: setup complete ===")
print(f"KB classes present: {', '.join(kb_classes)}")
print(f"EntityRuler patterns ready: {len(entityruler_patterns)}")
print(f"Alias index entries: {len(alias_index)}")
print("Label↔class mappings:", LABEL_TO_CLASS)
print("Context cues prepared for:", ", ".join(CONTEXT_CUES.keys()))
print("Iterator ready for raw_text.jsonl.")
print("mentions_buffer and debug_unlinked initialized (empty).")
